# Camada Silver — Transformações consolidadas

Este notebook unifica as transformações Silver dos cinco domínios da camada Bronze:

1. UF
2. Município
3. Meta Alfabetização Brasil
4. Meta Alfabetização UF
5. Alunos

A organização foi separada por responsabilidade, evitando repetição de funções e deixando o fluxo mais claro para manutenção e entrega acadêmica.


## 1. Configuração, imports e funções comuns

Responsabilidade desta seção:

- importar bibliotecas;
- configurar caminhos da Bronze e Silver;
- definir a data de processamento;
- concentrar funções reutilizáveis por todos os domínios.


In [16]:
from pathlib import Path
from datetime import date
import pandas as pd

# ------------------------------------------------------------
# Configurações gerais
# ------------------------------------------------------------

BRONZE_PATH = Path("../data/bronze")
SILVER_PATH = Path("../data/silver")
EXECUTION_DATE = date.today().isoformat()

print("Bronze path:", BRONZE_PATH.resolve())
print("Silver path:", SILVER_PATH.resolve())
print("Execution date:", EXECUTION_DATE)


# ------------------------------------------------------------
# Função: obter_arquivo_mais_recente
# ------------------------------------------------------------
# Procura arquivos Parquet dentro da pasta de uma tabela Bronze
# e retorna o arquivo mais recente.
# ------------------------------------------------------------

def obter_arquivo_mais_recente(caminho_tabela: Path) -> Path:
    arquivos = list(caminho_tabela.rglob("*.parquet"))

    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo Parquet encontrado em: {caminho_tabela}")

    return max(arquivos, key=lambda arquivo: arquivo.stat().st_mtime)


# ------------------------------------------------------------
# Função: padronizar_texto
# ------------------------------------------------------------
# Remove espaços extras no início e no fim de campos textuais.
# ------------------------------------------------------------

def padronizar_texto(valor):
    if pd.isna(valor):
        return valor

    return str(valor).strip()


# ------------------------------------------------------------
# Função: padronizar_sigla_uf
# ------------------------------------------------------------
# Padroniza a sigla da UF como texto em letras maiúsculas.
# ------------------------------------------------------------

def padronizar_sigla_uf(valor):
    if pd.isna(valor):
        return valor

    return str(valor).strip().upper()


# ------------------------------------------------------------
# Função: padronizar_codigo
# ------------------------------------------------------------
# Padroniza campos identificadores como texto.
#
# Motivo:
# Campos como id_aluno, id_escola e id_municipio são códigos,
# não medidas numéricas. Mantê-los como texto evita perda de
# formatação e facilita joins futuros.
# ------------------------------------------------------------

def padronizar_codigo(valor):
    if pd.isna(valor):
        return valor

    return str(valor).strip()


# ------------------------------------------------------------
# Função: validar_percentual
# ------------------------------------------------------------
# Cria flags de validação para campos percentuais.
#
# Regra:
# - valores entre 0 e 100 são válidos;
# - valores nulos também são considerados válidos nesta etapa.
# ------------------------------------------------------------

def validar_percentual(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    df_validado = df.copy()

    for coluna in colunas:
        if coluna in df_validado.columns:
            df_validado[f"flag_{coluna}_valido"] = (
                df_validado[coluna].isna()
                | df_validado[coluna].between(0, 100)
            )

    return df_validado


# ------------------------------------------------------------
# Função: salvar_silver
# ------------------------------------------------------------
# Salva um DataFrame tratado na camada Silver em formato Parquet.
#
# Estrutura:
# data/silver/{nome_tabela}/execution_date=YYYY-MM-DD/{nome_tabela}.parquet
# ------------------------------------------------------------

def salvar_silver(df: pd.DataFrame, nome_tabela: str) -> Path:
    output_dir = SILVER_PATH / nome_tabela / f"execution_date={EXECUTION_DATE}"
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{nome_tabela}.parquet"

    df.to_parquet(output_file, index=False)

    print(f"[OK] {nome_tabela} salvo em: {output_file}")
    print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")

    return output_file


Bronze path: C:\Projetos\fiap-tech-challenge-fase2\data\bronze
Silver path: C:\Projetos\fiap-tech-challenge-fase2\data\silver
Execution date: 2026-06-30


## 2. Carga das tabelas Bronze

Responsabilidade desta seção:

- localizar o arquivo Parquet mais recente de cada entidade Bronze;
- carregar os DataFrames originais que serão tratados na Silver.


In [17]:
# ------------------------------------------------------------
# Carregar bronze.uf
# ------------------------------------------------------------

arquivo_uf = obter_arquivo_mais_recente(BRONZE_PATH / "uf")
df_uf_bronze = pd.read_parquet(arquivo_uf)

print("bronze.uf:", arquivo_uf, df_uf_bronze.shape)
display(df_uf_bronze.head())


# ------------------------------------------------------------
# Carregar bronze.municipio
# ------------------------------------------------------------

arquivo_municipio = obter_arquivo_mais_recente(BRONZE_PATH / "municipio")
df_municipio_bronze = pd.read_parquet(arquivo_municipio)

print("bronze.municipio:", arquivo_municipio, df_municipio_bronze.shape)
display(df_municipio_bronze.head())


# ------------------------------------------------------------
# Carregar bronze.meta_alfabetizacao_brasil
# ------------------------------------------------------------

arquivo_meta_brasil = obter_arquivo_mais_recente(BRONZE_PATH / "meta_alfabetizacao_brasil")
df_meta_brasil_bronze = pd.read_parquet(arquivo_meta_brasil)

print("bronze.meta_alfabetizacao_brasil:", arquivo_meta_brasil, df_meta_brasil_bronze.shape)
display(df_meta_brasil_bronze.head())


# ------------------------------------------------------------
# Carregar bronze.meta_alfabetizacao_uf
# ------------------------------------------------------------

arquivo_meta_uf = obter_arquivo_mais_recente(BRONZE_PATH / "meta_alfabetizacao_uf")
df_meta_uf_bronze = pd.read_parquet(arquivo_meta_uf)

print("bronze.meta_alfabetizacao_uf:", arquivo_meta_uf, df_meta_uf_bronze.shape)
display(df_meta_uf_bronze.head())


# ------------------------------------------------------------
# Carregar bronze.alunos
# ------------------------------------------------------------

arquivo_alunos = obter_arquivo_mais_recente(BRONZE_PATH / "alunos")
df_alunos_bronze = pd.read_parquet(arquivo_alunos)

print("bronze.alunos:", arquivo_alunos, df_alunos_bronze.shape)
display(df_alunos_bronze.head())


bronze.uf: ..\data\bronze\uf\uf_processado.parquet (145, 19)


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,entidade_origem,modo_ingestao,data_ingestao_bronze
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,batch,2026-06-29 21:05:18.723760
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,batch,2026-06-29 21:05:18.723760
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,batch,2026-06-29 21:05:18.723760
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,batch,2026-06-29 21:05:18.723760
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,batch,2026-06-29 21:05:18.723760


bronze.municipio: ..\data\bronze\municipio\municipio_processado.parquet (23995, 19)


,ano,id_municipio,id_municipio_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,entidade_origem,modo_ingestao,data_ingestao_bronze
0,2023,1100031,Cabixi,2° ano do Ensino Fundamental,Municipal,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,batch,2026-06-29 21:05:18.632702
1,2023,1100072,Corumbiara,2° ano do Ensino Fundamental,Municipal,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,batch,2026-06-29 21:05:18.632702
2,2023,1100189,Pimenta Bueno,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,batch,2026-06-29 21:05:18.632702
3,2023,1101609,Theobroma,2° ano do Ensino Fundamental,Municipal,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,batch,2026-06-29 21:05:18.632702
4,2023,1101807,Vale do Paraíso,2° ano do Ensino Fundamental,Municipal,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,batch,2026-06-29 21:05:18.632702


bronze.meta_alfabetizacao_brasil: ..\data\bronze\meta_alfabetizacao_brasil\meta_alfabetizacao_brasil_processado.parquet (3, 14)


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,entidade_origem,modo_ingestao,data_ingestao_bronze
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00,meta_alfabetizacao_brasil,batch,2026-06-29 21:05:18.465511
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37,meta_alfabetizacao_brasil,batch,2026-06-29 21:05:18.465511
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00,meta_alfabetizacao_brasil,batch,2026-06-29 21:05:18.465511


bronze.meta_alfabetizacao_uf: ..\data\bronze\meta_alfabetizacao_uf\meta_alfabetizacao_uf_processado.parquet (81, 16)


,ano,sigla_uf,sigla_uf_nome,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,entidade_origem,modo_ingestao,data_ingestao_bronze
0,2024,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta_alfabetizacao_uf,batch,2026-06-29 21:05:18.575696
1,2023,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta_alfabetizacao_uf,batch,2026-06-29 21:05:18.575696
2,2024,SE,Sergipe,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84,meta_alfabetizacao_uf,batch,2026-06-29 21:05:18.575696
3,2023,SE,Sergipe,Pública,31.30,38.3,45.9,53.6,61.2,68.3,74.6,80.0,88.34,meta_alfabetizacao_uf,batch,2026-06-29 21:05:18.575696
4,2025,SE,Sergipe,Pública,50.00,38.0,46.0,54.0,61.0,68.0,75.0,80.0,87.00,meta_alfabetizacao_uf,batch,2026-06-29 21:05:18.575696


bronze.alunos: ..\data\bronze\alunos\alunos_processado.parquet (3867999, 16)


,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,entidade_origem,modo_ingestao,data_ingestao_bronze
0,2023,1301902,Itacoatiara,60000747,13012277,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,alunos,batch,2026-06-29 21:05:06.467497
1,2023,1502400,Castanhal,60001927,15039653,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,alunos,batch,2026-06-29 21:05:06.467497
2,2023,1505536,Parauapebas,60003091,15074844,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,alunos,batch,2026-06-29 21:05:06.467497
3,2023,2507507,João Pessoa,60011304,25036638,1,2° ano do Ensino Fundamental,Estadual,Ausente,Prova não preenchida,Não,NaN,NaN,alunos,batch,2026-06-29 21:05:06.467497
4,2023,2507507,João Pessoa,60011363,25036788,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,alunos,batch,2026-06-29 21:05:06.467497


## 3. Padronização das bases

Responsabilidade desta seção:

- criar cópias das tabelas Bronze;
- padronizar textos, códigos e siglas;
- converter colunas numéricas;
- remover duplicidades exatas.


In [ ]:
# ============================================================
# Padronização: UF
# ============================================================

df_uf_base = df_uf_bronze.copy()

colunas_numericas_uf = [
    "taxa_alfabetizacao",
    "media_portugues",
    "proporcao_aluno_nivel_0",
    "proporcao_aluno_nivel_1",
    "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3",
    "proporcao_aluno_nivel_4",
    "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6",
    "proporcao_aluno_nivel_7",
    "proporcao_aluno_nivel_8",
]

df_uf_base["ano"] = df_uf_base["ano"].astype("Int64")
df_uf_base["sigla_uf"] = df_uf_base["sigla_uf"].apply(padronizar_sigla_uf)
df_uf_base["sigla_uf_nome"] = df_uf_base["sigla_uf_nome"].apply(padronizar_texto)
df_uf_base["serie"] = df_uf_base["serie"].apply(padronizar_texto)
df_uf_base["rede"] = df_uf_base["rede"].apply(padronizar_texto)

for coluna in colunas_numericas_uf:
    if coluna in df_uf_base.columns:
        df_uf_base[coluna] = pd.to_numeric(df_uf_base[coluna], errors="coerce")

df_uf_base = df_uf_base.drop_duplicates().reset_index(drop=True)
print("df_uf_base:", df_uf_base.shape)


# ============================================================
# Padronização: Município
# ============================================================

df_municipio_base = df_municipio_bronze.copy()

colunas_numericas_municipio = colunas_numericas_uf.copy()

df_municipio_base["ano"] = df_municipio_base["ano"].astype("Int64")
df_municipio_base["id_municipio"] = df_municipio_base["id_municipio"].apply(padronizar_codigo)
df_municipio_base["id_municipio_nome"] = df_municipio_base["id_municipio_nome"].apply(padronizar_texto)
df_municipio_base["serie"] = df_municipio_base["serie"].apply(padronizar_texto)
df_municipio_base["rede"] = df_municipio_base["rede"].apply(padronizar_texto)

for coluna in colunas_numericas_municipio:
    if coluna in df_municipio_base.columns:
        df_municipio_base[coluna] = pd.to_numeric(df_municipio_base[coluna], errors="coerce")

df_municipio_base = df_municipio_base.drop_duplicates().reset_index(drop=True)
print("df_municipio_base:", df_municipio_base.shape)


# ============================================================
# Padronização: Meta Alfabetização Brasil
# ============================================================

df_meta_brasil_base = df_meta_brasil_bronze.copy()

colunas_metas = [
    "meta_alfabetizacao_2024",
    "meta_alfabetizacao_2025",
    "meta_alfabetizacao_2026",
    "meta_alfabetizacao_2027",
    "meta_alfabetizacao_2028",
    "meta_alfabetizacao_2029",
    "meta_alfabetizacao_2030",
]

colunas_numericas_meta = [
    "taxa_alfabetizacao",
    *colunas_metas,
    "percentual_participacao",
]

df_meta_brasil_base["ano"] = df_meta_brasil_base["ano"].astype("Int64")
df_meta_brasil_base["rede"] = df_meta_brasil_base["rede"].apply(padronizar_texto)

for coluna in colunas_numericas_meta:
    if coluna in df_meta_brasil_base.columns:
        df_meta_brasil_base[coluna] = pd.to_numeric(df_meta_brasil_base[coluna], errors="coerce")

df_meta_brasil_base = df_meta_brasil_base.drop_duplicates().reset_index(drop=True)
print("df_meta_brasil_base:", df_meta_brasil_base.shape)


# ============================================================
# Padronização: Meta Alfabetização UF
# ============================================================

df_meta_uf_base = df_meta_uf_bronze.copy()

df_meta_uf_base["ano"] = df_meta_uf_base["ano"].astype("Int64")
df_meta_uf_base["sigla_uf"] = df_meta_uf_base["sigla_uf"].apply(padronizar_sigla_uf)
df_meta_uf_base["sigla_uf_nome"] = df_meta_uf_base["sigla_uf_nome"].apply(padronizar_texto)
df_meta_uf_base["rede"] = df_meta_uf_base["rede"].apply(padronizar_texto)

for coluna in colunas_numericas_meta:
    if coluna in df_meta_uf_base.columns:
        df_meta_uf_base[coluna] = pd.to_numeric(df_meta_uf_base[coluna], errors="coerce")

df_meta_uf_base = df_meta_uf_base.drop_duplicates().reset_index(drop=True)
print("df_meta_uf_base:", df_meta_uf_base.shape)


# ============================================================
# Padronização: Alunos
# ============================================================

df_alunos_base = df_alunos_bronze.copy()

df_alunos_base["ano"] = df_alunos_base["ano"].astype("Int64")
df_alunos_base["id_municipio"] = df_alunos_base["id_municipio"].apply(padronizar_codigo)
df_alunos_base["id_municipio_nome"] = df_alunos_base["id_municipio_nome"].apply(padronizar_texto)
df_alunos_base["id_escola"] = df_alunos_base["id_escola"].apply(padronizar_codigo)
df_alunos_base["id_aluno"] = df_alunos_base["id_aluno"].apply(padronizar_codigo)
df_alunos_base["caderno"] = df_alunos_base["caderno"].apply(padronizar_texto)
df_alunos_base["serie"] = df_alunos_base["serie"].apply(padronizar_texto)
df_alunos_base["rede"] = df_alunos_base["rede"].apply(padronizar_texto)
df_alunos_base["presenca"] = df_alunos_base["presenca"].apply(padronizar_texto)
df_alunos_base["preenchimento_caderno"] = df_alunos_base["preenchimento_caderno"].apply(padronizar_texto)
df_alunos_base["alfabetizado"] = df_alunos_base["alfabetizado"].apply(padronizar_texto)

for coluna in ["proficiencia", "peso_aluno"]:
    if coluna in df_alunos_base.columns:
        df_alunos_base[coluna] = pd.to_numeric(df_alunos_base[coluna], errors="coerce")

df_alunos_base = df_alunos_base.drop_duplicates().reset_index(drop=True)
print("df_alunos_base:", df_alunos_base.shape)


df_uf_base: (145, 19)
df_municipio_base: (23995, 19)
df_meta_brasil_base: (3, 14)
df_meta_uf_base: (81, 16)


## 4. Criação das dimensões

Responsabilidade desta seção:

- criar tabelas dimensionais reutilizáveis;
- evitar repetição de cadastros territoriais e escolares nas tabelas fato.


In [ ]:
# ============================================================
# silver.dim_uf
# ============================================================

df_dim_uf = (
    df_uf_base[["sigla_uf", "sigla_uf_nome"]]
    .drop_duplicates()
    .sort_values("sigla_uf")
    .reset_index(drop=True)
)

df_dim_uf["data_processamento_silver"] = EXECUTION_DATE

print("silver.dim_uf:", df_dim_uf.shape)
display(df_dim_uf.head(30))


# ============================================================
# silver.dim_municipio
# ============================================================

df_dim_municipio = (
    df_municipio_base[["id_municipio", "id_municipio_nome"]]
    .drop_duplicates()
    .sort_values("id_municipio")
    .reset_index(drop=True)
)

df_dim_municipio["data_processamento_silver"] = EXECUTION_DATE

print("silver.dim_municipio:", df_dim_municipio.shape)
display(df_dim_municipio.head(30))


# ============================================================
# silver.dim_escola
# ============================================================

df_dim_escola = (
    df_alunos_base[["id_escola", "id_municipio", "id_municipio_nome"]]
    .drop_duplicates()
    .sort_values(["id_municipio", "id_escola"])
    .reset_index(drop=True)
)

df_dim_escola["data_processamento_silver"] = EXECUTION_DATE

print("silver.dim_escola:", df_dim_escola.shape)
display(df_dim_escola.head(30))


## 5. Criação das tabelas fato de resultado

Responsabilidade desta seção:

- criar fatos com resultados observados;
- armazenar indicadores como taxa de alfabetização, média de português e percentual de participação.


In [ ]:
# ============================================================
# silver.fato_resultado_uf
# ============================================================

df_fato_resultado_uf = df_uf_base[
    ["ano", "sigla_uf", "serie", "rede", "taxa_alfabetizacao", "media_portugues"]
].copy()

df_fato_resultado_uf = validar_percentual(df_fato_resultado_uf, ["taxa_alfabetizacao"])
df_fato_resultado_uf["data_processamento_silver"] = EXECUTION_DATE

df_fato_resultado_uf = (
    df_fato_resultado_uf
    .drop_duplicates()
    .sort_values(["ano", "sigla_uf", "serie", "rede"])
    .reset_index(drop=True)
)

print("silver.fato_resultado_uf:", df_fato_resultado_uf.shape)
display(df_fato_resultado_uf.head(20))


# ============================================================
# silver.fato_resultado_municipio
# ============================================================

df_fato_resultado_municipio = df_municipio_base[
    ["ano", "id_municipio", "serie", "rede", "taxa_alfabetizacao", "media_portugues"]
].copy()

df_fato_resultado_municipio = validar_percentual(df_fato_resultado_municipio, ["taxa_alfabetizacao"])
df_fato_resultado_municipio["data_processamento_silver"] = EXECUTION_DATE

df_fato_resultado_municipio = (
    df_fato_resultado_municipio
    .drop_duplicates()
    .sort_values(["ano", "id_municipio", "serie", "rede"])
    .reset_index(drop=True)
)

print("silver.fato_resultado_municipio:", df_fato_resultado_municipio.shape)
display(df_fato_resultado_municipio.head(20))


# ============================================================
# silver.fato_resultado_brasil
# ============================================================

df_fato_resultado_brasil = df_meta_brasil_base[
    ["ano", "rede", "taxa_alfabetizacao", "percentual_participacao"]
].copy()

df_fato_resultado_brasil = validar_percentual(
    df_fato_resultado_brasil,
    ["taxa_alfabetizacao", "percentual_participacao"]
)

df_fato_resultado_brasil["nivel_agregacao"] = "Brasil"
df_fato_resultado_brasil["data_processamento_silver"] = EXECUTION_DATE

df_fato_resultado_brasil = (
    df_fato_resultado_brasil
    .drop_duplicates()
    .sort_values(["ano", "rede"])
    .reset_index(drop=True)
)

print("silver.fato_resultado_brasil:", df_fato_resultado_brasil.shape)
display(df_fato_resultado_brasil.head(20))


# ============================================================
# silver.fato_resultado_meta_uf
# ============================================================

df_fato_resultado_meta_uf = df_meta_uf_base[
    ["ano", "sigla_uf", "rede", "taxa_alfabetizacao", "percentual_participacao"]
].copy()

df_fato_resultado_meta_uf = validar_percentual(
    df_fato_resultado_meta_uf,
    ["taxa_alfabetizacao", "percentual_participacao"]
)

df_fato_resultado_meta_uf["nivel_agregacao"] = "UF"
df_fato_resultado_meta_uf["data_processamento_silver"] = EXECUTION_DATE

df_fato_resultado_meta_uf = (
    df_fato_resultado_meta_uf
    .drop_duplicates()
    .sort_values(["ano", "sigla_uf", "rede"])
    .reset_index(drop=True)
)

print("silver.fato_resultado_meta_uf:", df_fato_resultado_meta_uf.shape)
display(df_fato_resultado_meta_uf.head(20))


## 6. Criação das tabelas fato de distribuição por nível

Responsabilidade desta seção:

- transformar colunas `proporcao_aluno_nivel_0` até `proporcao_aluno_nivel_8` em linhas;
- deixar a distribuição por nível em formato analítico.


In [ ]:
colunas_niveis = [
    "proporcao_aluno_nivel_0",
    "proporcao_aluno_nivel_1",
    "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3",
    "proporcao_aluno_nivel_4",
    "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6",
    "proporcao_aluno_nivel_7",
    "proporcao_aluno_nivel_8",
]


# ============================================================
# silver.fato_distribuicao_nivel_uf
# ============================================================

df_fato_distribuicao_nivel_uf = df_uf_base.melt(
    id_vars=["ano", "sigla_uf", "serie", "rede"],
    value_vars=colunas_niveis,
    var_name="nivel_alfabetizacao",
    value_name="proporcao_alunos"
)

df_fato_distribuicao_nivel_uf["nivel_alfabetizacao"] = (
    df_fato_distribuicao_nivel_uf["nivel_alfabetizacao"]
    .str.extract(r"(\d+)")
    .astype("Int64")
)

df_fato_distribuicao_nivel_uf = validar_percentual(
    df_fato_distribuicao_nivel_uf,
    ["proporcao_alunos"]
)

df_fato_distribuicao_nivel_uf["data_processamento_silver"] = EXECUTION_DATE

df_fato_distribuicao_nivel_uf = (
    df_fato_distribuicao_nivel_uf
    .drop_duplicates()
    .sort_values(["ano", "sigla_uf", "serie", "rede", "nivel_alfabetizacao"])
    .reset_index(drop=True)
)

print("silver.fato_distribuicao_nivel_uf:", df_fato_distribuicao_nivel_uf.shape)
display(df_fato_distribuicao_nivel_uf.head(20))


# ============================================================
# silver.fato_distribuicao_nivel_municipio
# ============================================================

df_fato_distribuicao_nivel_municipio = df_municipio_base.melt(
    id_vars=["ano", "id_municipio", "serie", "rede"],
    value_vars=colunas_niveis,
    var_name="nivel_alfabetizacao",
    value_name="proporcao_alunos"
)

df_fato_distribuicao_nivel_municipio["nivel_alfabetizacao"] = (
    df_fato_distribuicao_nivel_municipio["nivel_alfabetizacao"]
    .str.extract(r"(\d+)")
    .astype("Int64")
)

df_fato_distribuicao_nivel_municipio = validar_percentual(
    df_fato_distribuicao_nivel_municipio,
    ["proporcao_alunos"]
)

df_fato_distribuicao_nivel_municipio["data_processamento_silver"] = EXECUTION_DATE

df_fato_distribuicao_nivel_municipio = (
    df_fato_distribuicao_nivel_municipio
    .drop_duplicates()
    .sort_values(["ano", "id_municipio", "serie", "rede", "nivel_alfabetizacao"])
    .reset_index(drop=True)
)

print("silver.fato_distribuicao_nivel_municipio:", df_fato_distribuicao_nivel_municipio.shape)
display(df_fato_distribuicao_nivel_municipio.head(20))


## 7. Criação das tabelas fato de metas anuais

Responsabilidade desta seção:

- transformar as metas anuais de colunas para linhas;
- criar estrutura padronizada para comparação futura entre resultado observado e meta projetada.


In [ ]:
# ============================================================
# silver.fato_meta_anual_brasil
# ============================================================

df_fato_meta_anual_brasil = df_meta_brasil_base.melt(
    id_vars=["ano", "rede"],
    value_vars=colunas_metas,
    var_name="ano_meta",
    value_name="meta_alfabetizacao"
)

df_fato_meta_anual_brasil["ano_meta"] = (
    df_fato_meta_anual_brasil["ano_meta"]
    .str.extract(r"(\d{4})")
    .astype("Int64")
)

df_fato_meta_anual_brasil = validar_percentual(
    df_fato_meta_anual_brasil,
    ["meta_alfabetizacao"]
)

df_fato_meta_anual_brasil["nivel_agregacao"] = "Brasil"
df_fato_meta_anual_brasil["data_processamento_silver"] = EXECUTION_DATE

df_fato_meta_anual_brasil = (
    df_fato_meta_anual_brasil
    .drop_duplicates()
    .sort_values(["ano", "rede", "ano_meta"])
    .reset_index(drop=True)
)

print("silver.fato_meta_anual_brasil:", df_fato_meta_anual_brasil.shape)
display(df_fato_meta_anual_brasil.head(20))


# ============================================================
# silver.fato_meta_anual_uf
# ============================================================

df_fato_meta_anual_uf = df_meta_uf_base.melt(
    id_vars=["ano", "sigla_uf", "rede"],
    value_vars=colunas_metas,
    var_name="ano_meta",
    value_name="meta_alfabetizacao"
)

df_fato_meta_anual_uf["ano_meta"] = (
    df_fato_meta_anual_uf["ano_meta"]
    .str.extract(r"(\d{4})")
    .astype("Int64")
)

df_fato_meta_anual_uf = validar_percentual(
    df_fato_meta_anual_uf,
    ["meta_alfabetizacao"]
)

df_fato_meta_anual_uf["nivel_agregacao"] = "UF"
df_fato_meta_anual_uf["data_processamento_silver"] = EXECUTION_DATE

df_fato_meta_anual_uf = (
    df_fato_meta_anual_uf
    .drop_duplicates()
    .sort_values(["ano", "sigla_uf", "rede", "ano_meta"])
    .reset_index(drop=True)
)

print("silver.fato_meta_anual_uf:", df_fato_meta_anual_uf.shape)
display(df_fato_meta_anual_uf.head(20))


## 8. Criação da fato de aluno

Responsabilidade desta seção:

- criar a tabela fato no menor nível de granularidade da Silver;
- manter os indicadores individuais de presença, alfabetização, proficiência e peso;
- criar flags de qualidade para não descartar registros nesta camada.


In [ ]:
# ============================================================
# silver.fato_aluno_alfabetizacao
# ============================================================

df_fato_aluno_alfabetizacao = df_alunos_base[
    [
        "ano",
        "id_aluno",
        "id_escola",
        "id_municipio",
        "serie",
        "rede",
        "caderno",
        "presenca",
        "preenchimento_caderno",
        "alfabetizado",
        "proficiencia",
        "peso_aluno",
    ]
].copy()

# ------------------------------------------------------------
# Flags de qualidade
# ------------------------------------------------------------

df_fato_aluno_alfabetizacao["flag_id_aluno_valido"] = (
    df_fato_aluno_alfabetizacao["id_aluno"].notna()
    & (df_fato_aluno_alfabetizacao["id_aluno"] != "")
)

df_fato_aluno_alfabetizacao["flag_id_escola_valido"] = (
    df_fato_aluno_alfabetizacao["id_escola"].notna()
    & (df_fato_aluno_alfabetizacao["id_escola"] != "")
)

df_fato_aluno_alfabetizacao["flag_id_municipio_valido"] = (
    df_fato_aluno_alfabetizacao["id_municipio"].notna()
    & (df_fato_aluno_alfabetizacao["id_municipio"] != "")
)

df_fato_aluno_alfabetizacao["flag_proficiencia_valida"] = (
    df_fato_aluno_alfabetizacao["proficiencia"].isna()
    | (df_fato_aluno_alfabetizacao["proficiencia"] >= 0)
)

df_fato_aluno_alfabetizacao["flag_peso_aluno_valido"] = (
    df_fato_aluno_alfabetizacao["peso_aluno"].isna()
    | (df_fato_aluno_alfabetizacao["peso_aluno"] > 0)
)

df_fato_aluno_alfabetizacao["flag_presenca_preenchida"] = (
    df_fato_aluno_alfabetizacao["presenca"].notna()
    & (df_fato_aluno_alfabetizacao["presenca"] != "")
)

df_fato_aluno_alfabetizacao["flag_alfabetizado_preenchido"] = (
    df_fato_aluno_alfabetizacao["alfabetizado"].notna()
    & (df_fato_aluno_alfabetizacao["alfabetizado"] != "")
)

df_fato_aluno_alfabetizacao["data_processamento_silver"] = EXECUTION_DATE

df_fato_aluno_alfabetizacao = (
    df_fato_aluno_alfabetizacao
    .drop_duplicates()
    .sort_values(["ano", "id_municipio", "id_escola", "id_aluno"])
    .reset_index(drop=True)
)

print("silver.fato_aluno_alfabetizacao:", df_fato_aluno_alfabetizacao.shape)
display(df_fato_aluno_alfabetizacao.head(20))


## 9. Validações consolidadas

Responsabilidade desta seção:

- conferir volumetria das tabelas Silver;
- validar domínios principais;
- identificar registros inválidos pelas flags criadas.


In [ ]:
tabelas_silver = {
    "dim_uf": df_dim_uf,
    "fato_resultado_uf": df_fato_resultado_uf,
    "fato_distribuicao_nivel_uf": df_fato_distribuicao_nivel_uf,
    "dim_municipio": df_dim_municipio,
    "fato_resultado_municipio": df_fato_resultado_municipio,
    "fato_distribuicao_nivel_municipio": df_fato_distribuicao_nivel_municipio,
    "fato_resultado_brasil": df_fato_resultado_brasil,
    "fato_meta_anual_brasil": df_fato_meta_anual_brasil,
    "fato_resultado_meta_uf": df_fato_resultado_meta_uf,
    "fato_meta_anual_uf": df_fato_meta_anual_uf,
    "dim_escola": df_dim_escola,
    "fato_aluno_alfabetizacao": df_fato_aluno_alfabetizacao,
}

# ------------------------------------------------------------
# Volumetria geral
# ------------------------------------------------------------

print("=" * 80)
print("VOLUMETRIA DAS TABELAS SILVER")
print("=" * 80)

volumetria = []

for nome_tabela, df in tabelas_silver.items():
    volumetria.append(
        {
            "tabela": nome_tabela,
            "linhas": len(df),
            "colunas": len(df.columns),
        }
    )

df_volumetria_silver = pd.DataFrame(volumetria)
display(df_volumetria_silver)


# ------------------------------------------------------------
# Conferência de flags inválidas
# ------------------------------------------------------------

print("=" * 80)
print("REGISTROS INVÁLIDOS POR FLAG")
print("=" * 80)

linhas_flags = []

for nome_tabela, df in tabelas_silver.items():
    colunas_flags = [coluna for coluna in df.columns if coluna.startswith("flag_")]

    for coluna_flag in colunas_flags:
        linhas_flags.append(
            {
                "tabela": nome_tabela,
                "flag": coluna_flag,
                "qtd_invalidos": int((~df[coluna_flag]).sum()),
            }
        )

df_validacoes_flags = pd.DataFrame(linhas_flags)
display(df_validacoes_flags)


# ------------------------------------------------------------
# Conferências específicas
# ------------------------------------------------------------

print("UFs distintas em dim_uf:", df_dim_uf["sigla_uf"].nunique(dropna=True))
print("Municípios distintos em dim_municipio:", df_dim_municipio["id_municipio"].nunique(dropna=True))
print("Escolas distintas em dim_escola:", df_dim_escola["id_escola"].nunique(dropna=True))
print("Alunos distintos em fato_aluno_alfabetizacao:", df_fato_aluno_alfabetizacao["id_aluno"].nunique(dropna=True))

print("Anos de meta Brasil:", sorted(df_fato_meta_anual_brasil["ano_meta"].dropna().unique().tolist()))
print("Anos de meta UF:", sorted(df_fato_meta_anual_uf["ano_meta"].dropna().unique().tolist()))


## 10. Salvamento das tabelas Silver

Responsabilidade desta seção:

- salvar todos os DataFrames Silver em Parquet;
- manter a estrutura particionada por `execution_date`;
- conferir a leitura dos arquivos gravados.


In [ ]:
arquivos_silver = {}

for nome_tabela, df in tabelas_silver.items():
    arquivos_silver[nome_tabela] = salvar_silver(df, nome_tabela)


# ------------------------------------------------------------
# Conferência final dos arquivos salvos
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONFERÊNCIA FINAL DOS ARQUIVOS SILVER")
print("=" * 80)

for nome_tabela, arquivo in arquivos_silver.items():
    df_conferencia = pd.read_parquet(arquivo)

    print(f"{nome_tabela}: {arquivo}")
    print(f"Dimensão: {df_conferencia.shape}")
    display(df_conferencia.head())
